In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

silver_folder = 'abfss://silver@asterisktotle01.dfs.core.windows.net/PHHousing'

housing_silver = spark.read.format('delta').load(silver_folder + '/housing_silver')

dim_foreclosure = (
    housing_silver.select("sourceSlug", "sourceName").distinct()
    .withColumn("channel",
        F.when(F.lower("sourceSlug").contains("pagibig"), "Pag-IBIG")
         .otherwise("Bank (general)"))
    .withColumn("lenderSk", F.row_number().over(Window.orderBy("sourceSlug")))
    .select("lenderSk", "sourceSlug", "sourceName", "channel")
)
display(dim_foreclosure)


In [0]:
dim_lender_terms = spark.createDataFrame([
    ("Pag-IBIG",       0.90, 0.0575, 0.35, 10_000_000, 30, "official"),
    ("Bank (general)", 0.80, 0.0700, 0.30, None,        20, "estimated"),
], ["channel", "ltv_pct", "interest_rate", "dti_cap_pct",
    "max_loan_amount", "max_term_years", "source_confidence"])

display(dim_lender_terms)

In [0]:
silver_dim_geography = spark.read.format('delta').load(silver_folder + '/dim_geography')

gold_dim_geography = silver_dim_geography.withColumnRenamed('islandGroupCode', 'islandGroup')

display(gold_dim_geography)

In [0]:
dim_geography = spark.read.format('delta').load(silver_folder + '//geography_dim')
median_price_df = (
    housing_silver
    .join(dim_geography.select("cityCode"),
          housing_silver.geographyFk == dim_geography.cityCode, "left")
    .groupBy("geographyFk")
    .agg(
        F.expr("percentile_approx(price, 0.5)").alias("houseMedianPrice"),
        F.count("*").alias("listingCount")
    )
)

display(median_price_df)
